# Проект по Python: разведочный анализ данных по трафику аэропортов (2019–2022)

В этом ноутбуке сделан полноценный **EDA-анализ**:
- загрузка и объединение данных;
- очистка и первичная проверка;
- описательная статистика;
- анализ по годам, месяцам, аэропортам и странам;
- визуализация;
- статистические тесты;
- выводы и идеи для бизнес-логики.

Датасеты:
- `airport_traffic_2019.csv`
- `airport_traffic_2020.csv`
- `airport_traffic_2021.csv`
- `airport_traffic_2022.csv`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from scipy import stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


## 1. Загрузка данных

Сначала читаем все 4 CSV-файла и объединяем их в одну таблицу.  
Это удобно, потому что дальше все анализы можно делать уже на общем датафрейме.


In [ ]:
files = [
    'airport_traffic_2019.csv',
    'airport_traffic_2020.csv',
    'airport_traffic_2021.csv',
    'airport_traffic_2022.csv'
]

dfs = []
for file in files:
    df = pd.read_csv(file)
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
data['FLT_DATE'] = pd.to_datetime(data['FLT_DATE'], format='%d-%m-%y')

print('Размер таблицы:', data.shape)
display(data.head())
print('\nНазвания столбцов:')
print(list(data.columns))


## 2. Смысл столбцов

Основные поля:
- `YEAR` — год;
- `MONTH_NUM` — номер месяца;
- `MONTH_MON` — сокращенное название месяца;
- `FLT_DATE` — дата;
- `APT_ICAO` — ICAO-код аэропорта;
- `APT_NAME` — название аэропорта;
- `STATE_NAME` — страна;
- `FLT_DEP_1` — количество вылетов;
- `FLT_ARR_1` — количество прилетов;
- `FLT_TOT_1` — общее количество рейсов;
- `FLT_DEP_IFR_2`, `FLT_ARR_IFR_2`, `FLT_TOT_IFR_2` — показатели по IFR.

Сразу можно заметить, что переменные IFR могут быть заполнены не для всех строк.


## 3. Первичный контроль качества данных

Здесь проверяем:
- типы данных;
- пропуски;
- число уникальных аэропортов и стран;
- период наблюдений.


In [ ]:
print(data.info())

print('\nДоли пропусков:')
display((data.isna().mean() * 100).sort_values(ascending=False).round(2))

print('\nУникальных аэропортов:', data['APT_ICAO'].nunique())
print('Уникальных стран:', data['STATE_NAME'].nunique())
print('Минимальная дата:', data['FLT_DATE'].min())
print('Максимальная дата:', data['FLT_DATE'].max())


### Вывод по качеству данных
Что видно:
1. Основные поля без пропусков.
2. IFR-столбцы имеют много пропусков — это **нормально учитывать отдельно**, а не пытаться бездумно заменять на нули.
3. Данные покрывают период **с 2019-01-01 по 2022-12-31**.


## 4. Базовые описательные статистики

Сначала посмотрим на распределение числовых признаков.


In [ ]:
num_cols = ['FLT_DEP_1', 'FLT_ARR_1', 'FLT_TOT_1', 'FLT_DEP_IFR_2', 'FLT_ARR_IFR_2', 'FLT_TOT_IFR_2']
display(data[num_cols].describe().T)


## 5. Агрегация по годам

Очень важный блок: он показывает, как менялся общий трафик по годам.


In [ ]:
yearly = data.groupby('YEAR')[['FLT_DEP_1', 'FLT_ARR_1', 'FLT_TOT_1']].sum()
yearly['pct_change_tot'] = yearly['FLT_TOT_1'].pct_change() * 100
display(yearly)


### Ключевой смысл
По этой таблице обычно уже можно делать первые выводы:
- 2020 год резко проседает относительно 2019;
- 2021 показывает восстановление;
- 2022 продолжает восстановление, но не обязательно достигает уровня 2019 по всем сегментам.


In [ ]:
yearly['FLT_TOT_1'].plot(kind='bar')
plt.title('Общий объем рейсов по годам')
plt.ylabel('Количество рейсов')
plt.xlabel('Год')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. Анализ по месяцам

Теперь важно посмотреть сезонность.


In [ ]:
monthly = data.groupby(['YEAR', 'MONTH_NUM'])['FLT_TOT_1'].sum().reset_index()
monthly_pivot = monthly.pivot(index='MONTH_NUM', columns='YEAR', values='FLT_TOT_1')
display(monthly_pivot)

monthly_pivot.plot(marker='o')
plt.title('Динамика общего количества рейсов по месяцам')
plt.xlabel('Месяц')
plt.ylabel('Количество рейсов')
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Что здесь обычно видно
- у авиации есть сезонность;
- летние месяцы в среднем сильнее зимних;
- 2020 особенно резко проседает весной;
- после 2020 восстановление идет неравномерно.


## 7. Суточная агрегация по всей сети аэропортов

Чтобы проводить статистические тесты, удобно перейти на уровень **день -> суммарный трафик по всем аэропортам**.


In [ ]:
daily_total = data.groupby('FLT_DATE')['FLT_TOT_1'].sum().reset_index()
daily_total['YEAR'] = daily_total['FLT_DATE'].dt.year
daily_total['MONTH'] = daily_total['FLT_DATE'].dt.month

display(daily_total.head())

year_daily_desc = daily_total.groupby('YEAR')['FLT_TOT_1'].agg(['mean', 'median', 'std', 'min', 'max'])
display(year_daily_desc)


In [ ]:
for year, grp in daily_total.groupby('YEAR'):
    plt.hist(grp['FLT_TOT_1'], bins=30, alpha=0.6, label=year)

plt.title('Распределение суточного трафика по годам')
plt.xlabel('Количество рейсов за день')
plt.ylabel('Частота')
plt.legend()
plt.tight_layout()
plt.show()


## 8. Проверка нормальности и выбор статистического теста

Перед сравнением годов полезно проверить: можно ли считать распределение нормальным.  
Если нет, лучше использовать **непараметрические тесты**.


In [ ]:
normality_results = []

for year, grp in daily_total.groupby('YEAR'):
    stat, pvalue = stats.normaltest(grp['FLT_TOT_1'])
    normality_results.append([year, stat, pvalue])

normality_df = pd.DataFrame(normality_results, columns=['YEAR', 'statistic', 'pvalue'])
display(normality_df)


### Интерпретация
Если `p-value < 0.05`, гипотеза о нормальности отвергается.  
Значит, для сравнения нескольких годов лучше использовать **Kruskal-Wallis**, а для парных сравнений — **Mann-Whitney U**.


## 9. Сравнение годов: Kruskal-Wallis


In [ ]:
groups = [grp['FLT_TOT_1'].values for _, grp in daily_total.groupby('YEAR')]
kruskal_res = stats.kruskal(*groups)
print('Kruskal-Wallis statistic =', kruskal_res.statistic)
print('p-value =', kruskal_res.pvalue)


### Интерпретация
Если `p-value < 0.05`, распределения суточного трафика по годам статистически различаются.


## 10. Парные сравнения годов: Mann-Whitney U


In [ ]:
from itertools import combinations

pairwise_results = []
years = sorted(daily_total['YEAR'].unique())

for y1, y2 in combinations(years, 2):
    x = daily_total.loc[daily_total['YEAR'] == y1, 'FLT_TOT_1']
    y = daily_total.loc[daily_total['YEAR'] == y2, 'FLT_TOT_1']
    stat, pvalue = stats.mannwhitneyu(x, y, alternative='two-sided')
    pairwise_results.append([y1, y2, stat, pvalue, x.median(), y.median()])

pairwise_df = pd.DataFrame(pairwise_results, columns=['year_1', 'year_2', 'statistic', 'pvalue', 'median_1', 'median_2'])
display(pairwise_df)


## 11. Сезонность: сравнение лета и зимы

Сделаем отдельный тест:
- лето = июнь, июль, август;
- зима = декабрь, январь, февраль.


In [ ]:
daily_total['season'] = np.where(
    daily_total['MONTH'].isin([6, 7, 8]), 'summer',
    np.where(daily_total['MONTH'].isin([12, 1, 2]), 'winter', 'other')
)

summer = daily_total.loc[daily_total['season'] == 'summer', 'FLT_TOT_1']
winter = daily_total.loc[daily_total['season'] == 'winter', 'FLT_TOT_1']

season_test = stats.mannwhitneyu(summer, winter, alternative='two-sided')
print('Mann-Whitney U statistic =', season_test.statistic)
print('p-value =', season_test.pvalue)
print('Медиана лето =', summer.median())
print('Медиана зима =', winter.median())


## 12. Топ аэропортов

Теперь посмотрим на лидеров по общему числу рейсов.


In [ ]:
top_airports = (
    data.groupby(['APT_ICAO', 'APT_NAME'])['FLT_TOT_1']
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)

display(top_airports)

plt.barh(top_airports['APT_NAME'], top_airports['FLT_TOT_1'])
plt.title('Топ-15 аэропортов по общему числу рейсов')
plt.xlabel('Количество рейсов')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 13. Топ стран


In [ ]:
top_states = (
    data.groupby('STATE_NAME')['FLT_TOT_1']
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)

display(top_states)

plt.barh(top_states['STATE_NAME'], top_states['FLT_TOT_1'])
plt.title('Топ-15 стран по общему числу рейсов')
plt.xlabel('Количество рейсов')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 14. Изменение лидерства по годам

Сравним топы не только в сумме за 4 года, но и отдельно по каждому году.


In [ ]:
for year in sorted(data['YEAR'].unique()):
    print(f'\nТОП аэропортов за {year}')
    display(
        data[data['YEAR'] == year]
        .groupby(['APT_ICAO', 'APT_NAME'])['FLT_TOT_1']
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )


## 15. Анализ восстановления: 2022 к 2019

Очень сильный практический показатель — насколько аэропорт восстановился к 2022 году относительно 2019.


In [ ]:
airport_year = data.groupby(['APT_ICAO', 'APT_NAME', 'YEAR'])['FLT_TOT_1'].sum().reset_index()
airport_pivot = airport_year.pivot_table(index=['APT_ICAO', 'APT_NAME'], columns='YEAR', values='FLT_TOT_1')

airport_pivot['recovery_2022_vs_2019'] = airport_pivot[2022] / airport_pivot[2019]
large_airports = airport_pivot[airport_pivot[2019] >= 100000].copy()

best_recovery = large_airports[['recovery_2022_vs_2019']].sort_values('recovery_2022_vs_2019', ascending=False).head(10)
worst_recovery = large_airports[['recovery_2022_vs_2019']].sort_values('recovery_2022_vs_2019').head(10)

print('Лучшее восстановление:')
display(best_recovery)

print('Худшее восстановление:')
display(worst_recovery)


## 16. IFR-анализ

Так как IFR-поля заполнены не везде, анализируем их отдельно.


In [ ]:
ifr = data.dropna(subset=['FLT_TOT_IFR_2']).copy()
ifr = ifr[ifr['FLT_TOT_1'] > 0].copy()
ifr['ifr_share'] = ifr['FLT_TOT_IFR_2'] / ifr['FLT_TOT_1']

print('Доля строк с IFR-данными:', round(len(ifr) / len(data), 4))
display(ifr['ifr_share'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))


### Комментарий
Если доля IFR близка к 1, это означает, что IFR-рейсы составляют почти весь наблюдаемый трафик для соответствующей строки.  
Но здесь важно аккуратно интерпретировать аномалии, потому что местами отношение может быть больше 1 из-за особенностей агрегации или качества данных.


## 17. Поиск экстремальных дат


In [ ]:
print('Даты с максимальным трафиком:')
display(daily_total.nlargest(10, 'FLT_TOT_1')[['FLT_DATE', 'FLT_TOT_1']])

print('Даты с минимальным трафиком:')
display(daily_total.nsmallest(10, 'FLT_TOT_1')[['FLT_DATE', 'FLT_TOT_1']])


## 18. Интерактивная визуализация через Plotly


In [ ]:
fig = px.line(
    monthly,
    x='MONTH_NUM',
    y='FLT_TOT_1',
    color='YEAR',
    markers=True,
    title='Месячная динамика общего числа рейсов'
)
fig.show()


## 19. Возможные выводы для защиты проекта

Ниже — формулировки, которые можно говорить преподавателю:
1. Данные имеют хорошее качество по основным столбцам, но IFR-поля заполнены не полностью.
2. Главная особенность датасета — сильный структурный перелом в 2020 году.
3. После 2020 наблюдается восстановление, но оно неодинаково по аэропортам и странам.
4. Летняя сезонность выражена статистически значимо.
5. Крупнейшие аэропорты и страны формируют значительную часть всего объема.
6. Полезный бизнес-показатель — отношение трафика 2022 года к 2019 году.
7. Для сравнения годов лучше использовать непараметрические тесты, потому что нормальность распределений отвергается.


## 20. Как перевести это в бизнес-логику

Вот хорошие прикладные идеи:
- **Планирование ресурсов аэропорта**: персонал, слоты, обслуживание, безопасность.
- **Прогнозирование пиков** по месяцам и сезонам.
- **Оценка восстановления рынка** по аэропортам.
- **Сегментация аэропортов**: стабильные, быстро восстанавливающиеся, проблемные.
- **Оптимизация маршрутов и расписаний**.
- **Приоритезация инвестиций**: куда вкладываться в первую очередь.


## 21. Что можно добавить, чтобы проект выглядел еще сильнее

Если захочешь усилить проект, можно добавить:
1. STL-декомпозицию временного ряда.
2. Скользящее среднее по дням / неделям.
3. Кластеризацию аэропортов по профилю трафика.
4. Boxplot по годам и по месяцам.
5. Тепловую карту `месяц × год`.
6. Простую модель прогноза.
7. Проверку выбросов через IQR или z-score.


## 22. Итог

Этот проект уже не выглядит поверхностным, потому что в нем есть:
- очистка;
- описательная статистика;
- временной анализ;
- визуализация;
- статистические тесты;
- интерпретация;
- выход в бизнес-логику.

Именно это обычно и ждут от хорошего разведочного анализа данных.
